# Agent in ~50 lines

**Block 1, The AI Coding Agent Landscape**

You just watched GitHub Copilot fix a failing test. Copilot is a closed
product with a lot of UX polish around it; we can't open it up and look
inside. So instead, this notebook is a hand-rolled agent that solves the
*same* task with the *same* underlying mechanism, talking to the same
Claude model Copilot is using, through the same LiteLLM proxy.

It is not Copilot's code. It is the smallest honest reproduction of the
loop Copilot (and Claude Code, Cursor, Aider, Cline, OpenCode, and every
other coding agent) is built on. The real products differ in UX, polish,
sandboxing, and how many tools they expose, but the core loop is what's
below.

## 1. Setup

We need three things:

1. The **`litellm` Python SDK**, pointed at the workshop's **LiteLLM
   proxy**. Copilot in your Codespace is talking to the same proxy.
2. A **model alias** as configured on the proxy.
3. A **sandbox directory** so the agent's `write_file` and `run_bash`
   tools can't touch the rest of the repo.

LiteLLM normalizes everything to the OpenAI-style chat completions
format. That's a feature so that the same code below works against
Claude, GPT, Gemini, or any other model on the proxy. 

If `litellm.completion(...)` raises a 401, your `LITELLM_API_KEY` isn't
visible to the kernel, see [`docs/setup.md`](../../../docs/setup.md)
for how to add it as a Codespace secret.

In [ ]:
import json
import os
import subprocess
from pathlib import Path

import litellm

# Load a local .env if one exists (handy when running outside the
# Codespace). This is a no-op in the Codespace, where the gateway
# credentials arrive as injected secrets. load_dotenv does NOT override
# variables already in the environment, so Codespace secrets always win.
try:
    from dotenv import find_dotenv, load_dotenv

    load_dotenv(find_dotenv(usecwd=True))
except ModuleNotFoundError:
    pass

# Point the LiteLLM SDK at the LLM proxy server. The SDK does not
# auto-read these env vars, so we wire them onto the module once at the
# top: every subsequent litellm.completion() call uses them.
#
# The Codespace injects the gateway URL as LITELLM_BASE_URL (see
# .devcontainer/devcontainer.json); a local .env file can set the same
# variable when running outside the Codespace.
litellm.api_base = os.environ.get("LITELLM_BASE_URL")
litellm.api_key = os.environ.get("LITELLM_API_KEY")

if not litellm.api_base or not litellm.api_key:
    raise RuntimeError(
        "Missing gateway credentials. Set LITELLM_BASE_URL "
        "and LITELLM_API_KEY as Codespace secrets, export them in your shell, or "
        "put them in a .env file at the repo root. See docs/setup.md."
    )

# Model alias as configured on the proxy. Try "gpt-5" or
# "gemini-2.5-flash" later to see the same loop drive a different
# provider; if the proxy admin defined that alias, it just works.
MODEL = "claude-sonnet-4-6"

print("api_base =", litellm.api_base)
print("model    =", MODEL)

## 2. Sandboxing the agent

We're about to give a language model the ability to run `bash` and write
files. We want that power confined to the demo project, not pointed at
your home directory.

`SANDBOX` is the only path the agent can touch. `_safe_path` resolves a
relative path inside it and refuses anything that escapes via `..` or
absolute paths.

This is the simplest possible sandbox. Real agents (Copilot, Claude Code)
do much more, process isolation, syscall filters, network policies. But
the *idea* is the same: tools are gated.

In [ ]:
SANDBOX = (Path.cwd() / "starter").resolve()
assert SANDBOX.exists(), f"expected sandbox at {SANDBOX}"


def _safe_path(rel_path: str) -> Path:
    """Resolve rel_path inside SANDBOX, rejecting traversal."""
    p = (SANDBOX / rel_path).resolve()
    if p != SANDBOX and SANDBOX not in p.parents:
        raise ValueError(f"path {rel_path!r} escapes sandbox")
    return p


print("sandbox:", SANDBOX)

### Reset the demo

If you (or the agent) previously fixed `converters.py`, the cell below
puts it back to the buggy starting state so the rest of the notebook is
deterministic. Safe to re-run.

In [ ]:
BUGGY = '''"""Temperature unit conversions.

The Fahrenheit/Celsius converter contains a deliberate bug used by the
Block 1 demo. Do not "fix" it outside the demo flow.
"""


def fahrenheit_to_celsius(f: float) -> float:
    """Convert a temperature in degrees Fahrenheit to degrees Celsius."""
    return (f - 32) * 9 / 5


def celsius_to_fahrenheit(c: float) -> float:
    """Convert a temperature in degrees Celsius to degrees Fahrenheit."""
    return c * 9 / 5 + 32


def kelvin_to_celsius(k: float) -> float:
    """Convert a temperature in Kelvin to degrees Celsius."""
    return k - 273.15
'''

(SANDBOX / "src" / "sci_units" / "converters.py").write_text(BUGGY)
print("reset converters.py to buggy starting state")

## 3. The three tools we'll give the agent

A coding agent needs to *do things*, not just describe them. We give it
three primitives, read a file, write a file, run a shell command,
and that's enough to investigate, edit, and verify a code change.

Note that these are simple python definitions.

This is the same idea as Copilot's "agent mode": you'll see those exact
three capabilities (and a few more like web search) under the hood.

In [ ]:
def read_file(path: str) -> str:
    """Return the UTF-8 text contents of a sandboxed project file."""
    return _safe_path(path).read_text()


def write_file(path: str, content: str) -> str:
    """Overwrite a sandboxed file with `content`, creating parent dirs as needed."""
    p = _safe_path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(content)
    return f"wrote {len(content)} chars to {path}"


def run_bash(command: str) -> str:
    """Run a bash command inside SANDBOX and return its exit code, stdout, and stderr."""
    result = subprocess.run(
        ["bash", "-lc", command],
        cwd=SANDBOX,
        capture_output=True,
        text=True,
        timeout=60,
    )
    return (
        f"exit_code={result.returncode}\n"
        f"--- stdout ---\n{result.stdout}"
        f"--- stderr ---\n{result.stderr}"
    )


# Name -> handler lookup used by the agent loop to dispatch tool calls
# coming back from the model (see `run_agent` below).
TOOL_HANDLERS = {
    "read_file": read_file,
    "write_file": write_file,
    "run_bash": run_bash,
}

## 4. Tool schemas: what the model sees

The model can't introspect Python. We have to tell it, in JSON Schema,
what each tool is called, what it does, and what arguments it takes.

LiteLLM uses the **OpenAI tool-calling format**: each tool is wrapped
in a `{"type": "function", "function": {...}}` envelope. The actual
schema (under `parameters`) is plain JSON Schema, identical to what
you'd write for any other tool-using agent.

This is the **contract**. The model uses these descriptions to decide
*which* tool to call and *with what arguments*. Good descriptions
matter, vague schemas produce vague tool use.

In [ ]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": (
                "Read a UTF-8 text file from the project. "
                "Use this to inspect source code, tests, configs, or AGENTS.md."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {
                        "type": "string",
                        "description": "Project-relative path, e.g. 'src/sci_units/converters.py'",
                    },
                },
                "required": ["path"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "write_file",
            "description": (
                "Overwrite a UTF-8 text file with new content. Use after you have "
                "decided on the change you want to make. The whole file is replaced."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {"type": "string", "description": "Project-relative path"},
                    "content": {"type": "string", "description": "Full new file contents"},
                },
                "required": ["path", "content"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "run_bash",
            "description": (
                "Run a bash command from the project root. Useful for `pytest`, "
                "`ls`, `cat`, `grep`. Times out at 60 seconds."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "command": {"type": "string", "description": "Bash command line"},
                },
                "required": ["command"],
            },
        },
    },
]

## 5. The system prompt (and project memory)

This is where **`AGENTS.md`** comes in. Real agents like Claude Code and
Cursor automatically load `AGENTS.md` into the model's context so the
agent inherits project-specific guidance (style, conventions, layout).

We do the same thing manually below: read the `AGENTS.md` next to this
notebook and paste it into the system message. (In the OpenAI chat
format, the system prompt is just the first message in the list.)
That single mechanic is what "project memory" actually means.

In [ ]:
AGENTS_MD = (Path.cwd() / "AGENTS.md").read_text()

SYSTEM_PROMPT = f"""You are a careful coding agent working inside a small Python project.

You have three tools: read_file, write_file, run_bash. All paths are relative
to the project root (the directory containing pyproject.toml).

Your job: investigate problems, propose minimal fixes, and verify by running
the test suite. Keep changes small. Always re-run tests after editing.

When you are done, reply with a short plain-text summary of what you did.

Project memory (AGENTS.md):
---
{AGENTS_MD}
---
"""

print(SYSTEM_PROMPT)

## 6. The agent loop

Here's the whole trick. It's a `for` loop:

1. Send the conversation so far to the model.
2. If the model returned `tool_calls`, **execute** each one and append the
   results back as `role: tool` messages.
3. If the model returned only text (no `tool_calls`), we're done.
4. Repeat (with a step cap so an agent can't loop forever).

That's it. No magic. Every coding agent, Copilot, Claude Code, Cursor,
is some elaboration of this loop with more tools, better UX, and
production hardening.

Note the OpenAI-format LiteLLM exposes:

- Tool arguments arrive as a **JSON string** in `tc.function.arguments`,
  not a dict. We `json.loads` it before calling the handler.
- The assistant's full message (including its `tool_calls`) goes back
  into `messages` so the model sees its own decisions on the next turn.
- Tool results have `role: "tool"` and a `tool_call_id` linking back to
  the call they're answering.

In [ ]:
def run_agent(user_prompt: str, max_steps: int = 10) -> None:
    """Drive the model in a tool-use loop until it stops calling tools."""
    # The conversation is just a growing list of dicts in OpenAI chat
    # format. The system prompt comes first; everything the model and
    # tools say later gets appended to this same list.
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]

    # Hard cap on turns so a confused model can't burn the budget by
    # looping forever. Real agents add token/cost ceilings on top.
    for step in range(1, max_steps + 1):
        response = litellm.completion(
            model=MODEL,
            messages=messages,
            tools=TOOLS,
            max_tokens=4096,
        )
        msg = response.choices[0].message
        finish = response.choices[0].finish_reason

        print(f"\n--- step {step}  (finish_reason={finish}) ---")
        if msg.content:
            print(f"[text]\n{msg.content}")
        for tc in msg.tool_calls or []:
            print(f"[tool_call] {tc.function.name}({tc.function.arguments[:200]})")

        # Append the assistant's full message (text + tool_calls) back into
        # the conversation so the model sees its own decisions next turn.
        messages.append(msg.model_dump(exclude_none=True))

        # Termination condition: the model produced a plain text reply
        # with no tool calls, i.e. it thinks the task is done.
        if not msg.tool_calls:
            print("\n=== agent finished ===")
            return

        # Dispatch every tool call the model requested this turn. The
        # model can fan out multiple calls in a single response, so this
        # is a loop, not a single call.
        for tc in msg.tool_calls:
            try:
                # Arguments arrive as a JSON *string* in the OpenAI
                # format, never a dict. Parse, then splat into the
                # Python handler registered in TOOL_HANDLERS.
                args = json.loads(tc.function.arguments)
                output = TOOL_HANDLERS[tc.function.name](**args)
            except Exception as e:
                # Surface the failure back to the model as the tool
                # result instead of crashing the loop, the model can
                # often recover (retry with different args, give up
                # cleanly, etc.).
                output = f"ERROR: {type(e).__name__}: {e}"
            preview = output if len(output) <= 300 else output[:300] + "..."
            print(f"[tool_result]\n{preview}")
            # tool_call_id links this result back to the specific call
            # the model made. Required by the API when there were
            # multiple parallel tool calls in the same assistant turn.
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": output})

    print(f"\n=== hit max_steps={max_steps} without finishing ===")

## 7. Run it

We give the agent the same prompt the live Copilot demo received: *"there
is a failing test, find and fix it."*

Watch the trace below. You'll see the model:

1. Call `run_bash` to run pytest.
2. Read the failing test.
3. Read the implementation.
4. Reason about the bug.
5. Call `write_file` with the corrected code.
6. Re-run pytest.
7. Reply with a summary and stop.

The exact step-by-step varies, that's *also* part of the lesson. Agents
plan based on what they observe, so two runs against the same prompt can
look slightly different.

In [ ]:
run_agent(
    "There is a failing test in this project. "
    "Find it, understand the bug, fix it, and verify with pytest. "
    "Use the smallest possible change."
)

## 8. What we just saw

Two things, in order:

1. A few hundred milliseconds of model latency per turn.
2. A loop. That's it.

The loop is the entire mechanism. Everything else AI coding tools sell
you, agent mode, planning panes, slash commands, custom modes, is a
**user-experience layer** on top of this loop, plus more tools, plus
project memory, plus carefully tuned system prompts.

*All coding agents work the same way under the hood.*

> Try it: change `MODEL` to another alias the proxy exposes (ask an
> instructor what's available, likely a GPT or a Gemini). Re-run the
> agent against the same prompt. The loop, the tools, and the schemas
> don't change. The model does. 

## 9. Where the Block 1 anatomy lives in this code

Every concept on the "anatomy of a coding agent" slide maps to a few
lines you just executed:

| Anatomy concept | Where it is in this notebook |
|---|---|
| **LLM backbone** | `litellm.api_base = ...` and `MODEL = ...`. Swappable in one line. |
| **Tool use** | `TOOLS = [...]` (OpenAI-format schemas) + the `tool_calls` branch in `run_agent` |
| **Agent loop / planning** | The `for step in range(max_steps): ...` loop |
| **Project memory (`AGENTS.md`)** | `AGENTS_MD = (Path.cwd() / "AGENTS.md").read_text()` pasted into the system message |
| **MCP servers** | Would replace our hand-written `TOOL_HANDLERS` with a generic MCP client. The schemas would come from the MCP server instead of being hard-coded. |
| **Skills / prompts / agents** | Templated user-prompt prefixes invoked by name (e.g., `"/review-pr"` could expand to a longer prompt that gets prepended to `messages`). |

When you see a feature in Copilot, Claude Code, or Cursor and want to
understand it, ask: *which of these six things is it?* That mental model
will outlive any of the products.

## 10. Bridge to Block 2

One question this notebook glosses over: **why does the model know to
call `run_bash` instead of just describing what it would do?**

Nothing in the JSON schemas told it "you're allowed to use these." We
just gave it a list of tools and it correctly chose to *use* them, in a
sensible *order*, with sensible *arguments*. And we just hand-waved that
this same loop works for Claude, GPT, and Gemini, even though those
models were trained by three different companies on different data.

That common behavior didn't fall out of next-token prediction on web
text. It was **trained in** during the post-training phase: instruction
tuning, tool-use fine-tuning, and reinforcement learning. Convergent
post-training is why model swaps are even possible. That's Block 2.